In [14]:
import pandas as pd

df = pd.read_csv('../data/db/events.csv')

df.head()

,id,date_start,date_end,event
0,00dc6acf-fc00-4055-bec0-76cdfa15ddf6,2000-01-01,NaN,Деноминация белорусского рубля;
1,00dc6acf-fc00-4122-ab64-cdc14edaf980,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,00dc6acf-fc00-4865-87e5-ed4168d72a97,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,00dc6ff6-7800-43ae-ac8e-b2b8989debc2,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...
4,00dc751c-7400-4aa0-a97f-4141d66267a2,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [7]:
import spacy
from spacy.matcher import DependencyMatcher

nlp = spacy.load("ru_core_news_lg")
dm = DependencyMatcher(nlp.vocab)

# 1️⃣ Ввели / наложили санкции
pattern_action = [
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"LEMMA": {"IN": ["ввести", "наложить"]}}},
    {"LEFT_ID": "verb", "REL_OP": ">>", "RIGHT_ID": "sanction", "RIGHT_ATTRS": {"LEMMA": "санкция"}},
]

lemmas_pkg = [
    "принять", "принят", "принятие",
    "утвердить", "утвержден", "утверждение",
    "одобрить", "одобрен", "одобрение"
]

pattern_pkg_any = [
    {
        "RIGHT_ID": "root",
        "RIGHT_ATTRS": {
            "LEMMA": {"IN": lemmas_pkg}
        },
    },
    {
        "LEFT_ID": "root",
        "REL_OP": ">>",  # допускаем промежуточные звенья (числительные, прилагательные)
        "RIGHT_ID": "package",
        "RIGHT_ATTRS": {"LEMMA": "пакет"},
    },
    {
        "LEFT_ID": "package",
        "REL_OP": ">>",
        "RIGHT_ID": "sanction",
        "RIGHT_ATTRS": {"LEMMA": "санкция"},
    },
]

dm.add("SANCTION", [pattern_action, pattern_pkg_any])

# === Тест ===
text = """
Принят 9 пакет санкций против России.
"""

doc = nlp(text)

matches = dm(doc)
seen = set()
for mid, toks in matches:
    span = doc[min(toks): max(toks) + 1]
    key = tuple(sorted(toks))
    if key in seen:
        continue
    seen.add(key)
    print(f"{nlp.vocab.strings[mid]} → {span.text}")

SANCTION → Принят 9 пакет санкций


In [18]:
def has_sanction(text, matcher=dm):
    doc = nlp(text)
    matches = matcher(doc)
    return len(matches) > 0


df["is_sanction"] = df["event"].apply(has_sanction)

print(len(df[df["is_sanction"] == True]))
df

20


,id,date_start,date_end,event,is_sanction
0,00dc6acf-fc00-4055-bec0-76cdfa15ddf6,2000-01-01,NaN,Деноминация белорусского рубля;,False
1,00dc6acf-fc00-4122-ab64-cdc14edaf980,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н...",False
2,00dc6acf-fc00-4865-87e5-ed4168d72a97,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог...",False
3,00dc6ff6-7800-43ae-ac8e-b2b8989debc2,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...,False
4,00dc751c-7400-4aa0-a97f-4141d66267a2,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...,False
...,...,...,...,...,...
5642,01995a1e-fc00-42db-9627-ee5dfe740f57,2025-09-18,NaN,на Камчатке зафиксировано землетрясение магнит...,False
5643,0199646b-f400-47ff-bfe7-cf3d10687798,2025-09-20,NaN,проведение конкурса песни «Интервидение» в Мос...,False
5644,019973de-f800-4540-88c9-d9369d487b21,2025-09-23,NaN,Международный уголовный суд представил подтвер...,False
5645,01997e2b-7000-4b2b-b7d6-4a385a5b6932,2025-09-25,NaN,Парламент Кыргызстана объявил о самороспуске.,False


In [20]:
import pandas as pd
import os

def tag_sanction_events(df, event_tags_path: str):
    """
    Добавляет тэг SANCTIONS к событиям, где has_sanction == True.
    Перед выполнением запрашивает подтверждение в консоли.

    df — DataFrame с событиями (обязательно с колонками ['id', 'event', 'is_sanction'])
    event_tags_path — путь к файлу event_tags.csv
    """

    confirm = input(f"⚠️ Добавить тэг 'SANCTIONS' для событий с санкциями в '{event_tags_path}'? [y/n]: ").strip().lower()
    if confirm != "y":
        print("Операция отменена пользователем.")
        return

    sanctions_tag_code = "SANCTIONS"

    # === 🔹 Фильтруем события, где есть санкции ===
    sanction_events = df[df["is_sanction"] == True][["id"]].copy()

    if sanction_events.empty:
        print("⚠️ Не найдено событий с санкциями.")
        return

    sanction_events["tag_code"] = sanctions_tag_code

    # === 🔹 Загружаем или создаём event_tags.csv ===
    if os.path.exists(event_tags_path):
        event_tags = pd.read_csv(event_tags_path, encoding="utf-8-sig")
    else:
        print(f'Файла {event_tags_path} не существует. Создайте его и повторите попытку.')
        return

    # === 🔹 Добавляем новые связи ===
    new_records = pd.DataFrame({
        "event_id": sanction_events["id"],
        "tag_code": sanction_events["tag_code"]
    })

    # Удаляем дубликаты (если запись уже есть)
    combined = pd.concat([event_tags, new_records], ignore_index=True)
    combined = combined.drop_duplicates(subset=["event_id", "tag_code"])

    # === 🔹 Сохраняем обновлённый файл ===
    combined.to_csv(event_tags_path, index=False, encoding="utf-8-sig")

    print(f"✅ Добавлено/обновлено {len(new_records)} связей в '{event_tags_path}'.")

tag_sanction_events(df, '../data/db/event_tags.csv')

✅ Добавлено/обновлено 20 связей в '../data/db/event_tags.csv'.
